# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashelesc/flyrank_ml_internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Ranking Signal Analysis.**

I'm choosing this lane because it lets me test intuitions about what actually moves search performance instead of assuming them. Early checks on the starter dataset already showed a counter-intuitive result — keyword search volume barely predicts the traffic a page gets, while position tier clearly predicts click-through rate. That gap between what "feels" like a lever and what the data shows is a lever is exactly the kind of question I want to spend the next 7 weeks on: identifying which measurable content and search signals are real, actionable levers versus which are noise, so recommendations to a content/SEO team are backed by evidence rather than assumption.


In [8]:
import pandas as pd

df = pd.read_csv("content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(df["trend_direction"].value_counts())

Rows: 30,000
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** which measurable factors (search volume, position tier, word count, content type, staleness) are worth prioritizing when trying to recover a declining page.

**Who acts on it:** the content/SEO team, who use the signal ranking to decide where to spend limited editing and optimization effort.

**Cost of a wrong call:** if the report wrongly flags a weak signal — like search volume — as a strong lever, the team could spend weeks targeting high-volume keywords expecting a traffic lift that never arrives. That's wasted content budget and a missed opportunity to fix what actually drives clicks, such as improving position or refreshing stale content.


In [9]:
n_declining = (df["trend_direction"] == "down").sum()
n_high_volume_declining = ((df["trend_direction"] == "down") & (df["search_volume"] > df["search_volume"].median())).sum()

print(f"Declining pages in this dataset: {n_declining:,}")
print(f"Of those, {n_high_volume_declining:,} have above-median search_volume")
print("-> this is roughly how many pages a wrong 'chase search volume' recommendation would misdirect effort toward.")

Declining pages in this dataset: 16,262
Of those, 4,667 have above-median search_volume
-> this is roughly how many pages a wrong 'chase search volume' recommendation would misdirect effort toward.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
# Number 1: does search volume predict impressions?
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.\n")

# Number 2: does position tier predict CTR?
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Average CTR by position tier:")
print(ctr_by_pos.round(4).to_string())
print()

# Number 3: is word count a lever for declining vs growing pages?
wc = df.groupby("trend_direction")["word_count"].median()
print("Median word count by trend direction:")
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have almost the same median word count -> length is not the lever.")

Correlation between search_volume and impressions_90d: 0.001
Near zero -> keyword search volume barely predicts the traffic a page actually gets.

Average CTR by position tier:
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

Median word count by trend direction:
trend_direction
down      2909.0
flat      2698.0
new       2239.0
stable    2912.0
up        2848.0

'down' vs 'up' pages have almost the same median word count -> length is not the lever.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can say:** these numbers are observed associations in the starter dataset — a near-zero correlation between search volume and impressions, a clear directional drop in CTR as position tier worsens, and near-identical word counts across declining and growing pages. These are decision-support signals: they tell the content team where the data currently points, and where it doesn't.

**What I can't say:** none of this is causal proof. A directional pattern (better position tier → higher CTR) is not the same as proving that *changing* a page's position *causes* CTR to rise for that specific page — other factors could explain both. This analysis also does not predict how Google's ranking algorithm works or will behave; it only describes patterns already present in this dataset over this time window. Any recommendation drawn from it should be framed as "the data suggests" or "this is associated with," never as a guarantee.

In [7]:
print(f"Sample size: {len(df):,} pages")
print(f"Correlation search_volume vs impressions_90d: {df['search_volume'].corr(df['impressions_90d']):.3f}  (weak -> directional only, not causal)")

# Privacy check: confirm no client-identifying columns are present
sensitive = [c for c in df.columns if any(k in c.lower() for k in ["client", "url", "query", "name"])]
print(f"Columns needing a privacy check before sharing: {sensitive}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.